# Introduction to Reinforcement Learning
In this module, we will explore the fundamentals of Reinforcement Learning (RL), a branch of machine learning focused on how agents should take actions in an environment to maximize cumulative reward over time. 

> __Learning Objectives__
> By the end of this module, you will be able to define and demonstrate mastery of the following key concepts:

> * __Exploration vs. Exploitation__: In reinforcement learning, the exploration vs. exploitation trade‐off forces an agent to balance trying new actions to discover potentially better rewards (exploration) against leveraging its current knowledge to maximize immediate payoff (exploitation). Striking the right balance is crucial for learning an optimal policy that performs well both now and in the long run.
> * __Q-learning__: A model-free, off-policy algorithm that learns the optimal action-value function by iteratively updating its value estimates using observed rewards and the best available future estimates, thereby converging to an optimal policy without requiring any model of the environment.

At its core, RL is about learning from doing: an agent observes the state of its environment, takes actions, i.e., makes decisions, receives rewards, and updates its knowledge to improve future decision-making. Let’s get started!
___

## General Reinforcement Learning Problem
Suppose we have an agent that can be in a state $s \in \mathcal{S}$ and can take an action $a \in \mathcal{A}$. After taking action $a$ in state $s$, the agent receives a reward $r$. But how does the agent learn to choose the best possible action in each state to maximize its cumulative reward over time?

<div>
    <center>
        <img src="figs/Fig-Schematic-RL.svg" width="580"/>
    </center>
</div>

In reinforcement learning, an agent interacts with an environment by observing its current state $s \in \mathcal{S}$, selecting an action $a \in \mathcal{A}$, and receiving a reward that influences its future decisions. We'll explore three distinct approaches to this problem:

* __Bandit algorithms__ operate in stateless environments. On each round, they explore different actions to estimate their rewards and adapt their action-selection strategy based on the outcomes.
* __Multiplicative weights__ also adapt action probabilities based on past performance, but they do so in a principled way that guarantees the algorithm performs nearly as well as the best fixed action in hindsight—even in changing environments.
* __Q-learning__ is a value-based method that estimates the long-term value of each state-action pair, enabling the agent to learn optimal behavior in environments with temporal and sequential dynamics.


These approaches highlight different strategies for learning from interaction, but they all must balance a fundamental challenge in reinforcement learning: the tradeoff between exploring new actions to gather information and exploiting known actions to maximize reward.

___

## Exploration vs. Exploitation
In reinforcement learning, the problem on the surface is deceptively simple: an agent is in a state $s$ and can take an action $a\in A_{s}$, where $A_{s}$ is the set of actions currently available to the agent. The agent chooses an action $a$, implements it, and receives a reward $r$ and transitions to a new state $s^{\prime}$. The goal is to learn a policy that maximizes the cumulative reward over time, i.e., the best possible action in each state.

The problem is more complex than it appears. The agent must make decisions based on incomplete information and balance two competing objectives: exploration and exploitation. The exploration vs. exploitation trade-off is a fundamental challenge that agents must navigate:
1. **Exploration**: Trying new actions to discover potentially better rewards. This is essential for learning about the environment and finding optimal policies. Taking random actions or actions that have not been tried often to gather information about their outcomes and rewards is an example of exploration.
2. **Exploitation**: Leveraging current knowledge to maximize immediate payoff. This involves choosing actions that have previously yielded high rewards based on the agent's experience. If the agent only exploits, it may miss out on discovering better actions that could yield higher rewards in the long run.

Striking the right balance between exploration and exploitation is crucial for learning an optimal policy that performs well both now and in the long run. If an agent explores too much, it may miss out on immediate rewards; if it exploits too much, it may fail to discover better long-term strategies.

The exploration-exploitation trade-off is often formalized in algorithms that guide the agent's decision-making process. These algorithms provide principled strategies for managing the trade-off, ensuring that the agent can learn effectively while maximizing its cumulative reward over time.
___

## Concept Review: Value Functions, Value Iteration, and Policy Functions
Before we dive into specific reinforcement learning algorithms, let's quickly review some key concepts from Markov decision processes (MDPs) that will be important for understanding how these algorithms work.

Value iteration is a dynamic programming algorithm that computes the optimal value function $U^{*}(s)$ by iteratively applying the Bellman backup operation:

$$
\begin{equation*}
U_{k+1}(s) = \max_{a\in\mathcal{A}}\left(\underbrace{R(s,a)}_{\text{= now}} + 
\gamma\;\overbrace{\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot{U}_{k}(s^{\prime})}^{\text{= future}}\right)
\end{equation*}
$$

As $k \to \infty$, the value function is __guaranteed to converge__ for $0\leq\gamma<1$ such that $U_k(s) \to U^{*}(s)$ [1,2,5]. The optimal value function represents the maximum expected cumulative discounted reward achievable from each state under the best possible policy. Let's develop the value iteration algorithm for computing the optimal value function $U^{*}(s)$ and policy $\pi^{*}(s)$.

### Algorithm: Value Iteration

__Initialize__: Given an MDP with state space $\mathcal{S}$, action space $\mathcal{A}$, reward function $R(s,a)$, transition model $T\left(s^{\prime}\,|\,s,a\right)$, discount factor $\gamma$, tolerance parameter $\epsilon$, and maximum number of iterations $T$. Initialize the iteration counter $k\gets 0$, the initial value function $U_{0}(s) \gets 0$ for all $s \in \mathcal{S}$, and $\texttt{converged}\gets\texttt{false}$.

While $\texttt{converged}$ is $\texttt{false}$ __do__:
1. For each state $s \in \mathcal{S}$, compute the updated value:
   $$U_{k+1}(s) \gets \max_{a\in\mathcal{A}}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot{U}_{k}(s^{\prime})\right)$$
2. Check for convergence:
    - If $\max_{s\in\mathcal{S}} \left|U_{k+1}(s) - U_{k}(s)\right| \leq \epsilon$, then set $\texttt{converged}\gets\texttt{true}$ and $U^{*}\gets{U}_{k+1}$.
    - If $\max_{s\in\mathcal{S}} \left|U_{k+1}(s) - U_{k}(s)\right| > \epsilon$, update $k\gets{k+1}$ and $U_{k}\gets{U}_{k+1}$.
3. Update the $\texttt{converged}$ flag:
    - If $k\geq{T}$, then set $\texttt{converged}\gets\texttt{true}$ and $U^{*}\gets{U}_{k+1}$. Notify the caller that the maximum iteration limit was reached without convergence.

__Extract Policy__: For each state $s \in \mathcal{S}$, compute:
$$\pi^{*}(s) \gets \arg\max_{a\in\mathcal{A}}\left(\underbrace{R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}}T\left(s^{\prime}\,|\,s,a\right)\cdot{U^{*}}(s^{\prime})}_{\text{state-action-value } Q(s,a)}\right)$$

### Computational Complexity

Value iteration requires $O(|\mathcal{S}|^2 \cdot |\mathcal{A}|)$ operations per iteration. For each of the $|\mathcal{S}|$ states, we must evaluate $|\mathcal{A}|$ actions, and each action evaluation requires summing over $|\mathcal{S}|$ possible next states. The number of iterations until convergence depends on the discount factor $\gamma$ and the tolerance $\epsilon$, with convergence rate proportional to $\gamma$.

This complexity makes value iteration tractable for problems with thousands of states but challenging for very large discrete state spaces. In such cases, function approximation or sampling-based methods become necessary.
___


<div>
    <center>
        <img src="figs/Fig-Q-Schematic.svg" width="580"/>
    </center>
</div>

## Q-Learning Theory
Q-learning iteratively estimates the state action-value function $Q(s, a)$ by conducting repeated experiments $t=1,2,\ldots$ in the world $\mathcal{W}$. 
In each experiment, an agent in state $s\in\mathcal{S}$ takes action $a\in\mathcal{A}$, receives a reward $r$, and (potentially) transitions to a new state $s^{\prime}$. After each experiment $t$, the agent updates its estimate of $Q(s, a)$ using the update rule:
$$
\begin{equation*}
Q_{t+1}(s,a)\leftarrow{Q_{t}(s,a)}+\alpha_{t}\cdot\underbrace{\left(r+\gamma\cdot\max_{a^{\prime}\in\mathcal{A}}Q_{t}(s^{\prime},a^{\prime}) - Q_{t}(s,a)\right)}_{\text{new information}}\quad{t = 1,2,3,\ldots}
\end{equation*}
$$
where $0<\alpha_{t} <{1}$ is the learning rate parameter at time $t$, and $0<\gamma<{1}$ is the discount factor. 
We estimate the policy function $\pi:\mathcal{S}\rightarrow\mathcal{A}$ by selecting the action $a$ that maximizes $Q(s,a)$ at each state $s$:
$$
\begin{equation*}
\pi(s) = \arg\max_{a\in\mathcal{A}}Q(s,a)
\end{equation*}
$$

### Algorithm
Initialize $Q(s,a)$ arbitrarily for all $s\in\mathcal{S}$, and $a\in\mathcal{A}$.
Set the hyperparameters: learning rate $\alpha_{t}$, the discount factor $\gamma$, the exploration rate $\epsilon_{t}$, and the convergence tolerance $\delta$.

For $s\in\mathcal{S}$
1. Initialize the time $t\gets{1}$
2. While not converged:
    1. Roll a random number $p\in[0,1]$.
    2. If $p\leq\epsilon_{t}$, choose a random (uniform) action $a_{t}\in\mathcal{A}$. Otherwise, choose a greedy action $a_{t} = \text{arg}\max_{a\in\mathcal{A}}{Q_{t}(s,a)}$.
    3. Take action $a_{t}$, observe the reward $r$ from the _world_ and transition to the next state $s^{\prime}$.
    4. Update the state-action-value function: $Q_{t+1}(s,a)\leftarrow{Q_{t}(s,a)}+\alpha_{t}\cdot\underbrace{\left(r+\gamma\cdot\overbrace{\max_{a^{\prime}\in\mathcal{A}}Q_{t}(s^{\prime},a^{\prime})}^{\text{one-step lookahead}} - Q_{t}(s,a)\right)}_{\text{new information}}$.
    5. Update the state $s\leftarrow{s^{\prime}}$, the time $t\leftarrow{t+1}$, the exploration rate $\epsilon_{t+1}\leftarrow\epsilon_{t}$, and the learning rate $\alpha_{t+1}\leftarrow\alpha_{t}$.
    6. Check for convergence. If the $Q(s,a)$ has bounded change $\lVert{Q_{t+1}(s,a) - Q_{t}(s,a)}\rVert\leq\delta$, then the algorithm has converged. Otherwise, continue.
3. End While
4. End For

### Convergence
Q-learning converges to the optimal policy under two key theoretical conditions (assuming the Markov property holds for the environment) [3,4,5,6,7]:
* __Learning rate decay__: The learning rate $\alpha_{t}$ must satisfy $\sum_{t=0}^\infty \alpha_t(s, a) = \infty$ and $\sum_{t=0}^\infty \alpha_t^2(s, a) < \infty$ for all state-action pairs, ensuring sufficient initial updates while stabilizing over time [7]. Setting $\alpha_{t+1} \gets \beta\alpha_{t}$ where $\beta<1$ is a common choice.
* __Infinite exploration__: All state-action pairs must be visited infinitely often [4]. This condition holds for $\epsilon$-greedy policies with persistent exploration, i.e., $\epsilon_{t} > 0\,\,\forall{t}$.

___

## Summary
In this module, we explored reinforcement learning fundamentals, focusing on how agents learn to make decisions through interaction with their environment.

> __Key Takeaways:__
>
> * **Exploration vs. exploitation trade-off:** Agents must balance trying new actions to discover better strategies against using known actions to maximize immediate rewards. This balance is essential for learning optimal policies.
> * **Value iteration for known models:** When the transition model is known, value iteration computes optimal policies by iteratively applying the Bellman backup operation until the value function converges.
> * **Q-learning for unknown models:** Q-learning learns optimal policies without requiring knowledge of the environment dynamics. It updates action-value estimates based on observed rewards and converges to the optimal policy under appropriate conditions.

Reinforcement learning enables agents to learn optimal behavior through trial and error in both known and unknown environments.
___

## References

1. **Bellman, R.** (1957). *Dynamic Programming*. Princeton University Press.

2. **Puterman, M. L.** (1994). *Markov Decision Processes: Discrete Stochastic Dynamic Programming*. John Wiley & Sons.

3. **Watkins, C. J. C. H., & Dayan, P.** (1992). Q-learning. *Machine Learning*, 8(3-4), 279-292.

4. **Watkins, C. J. C. H.** (1989). *Learning from Delayed Rewards*. PhD thesis, Cambridge University.

5. **Sutton, R. S., & Barto, A. G.** (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press.

6. **Bertsekas, D. P., & Tsitsiklis, J. N.** (1996). *Neuro-Dynamic Programming*. Athena Scientific.

7. **Jaakkola, T., Jordan, M. I., & Singh, S. P.** (1994). On the convergence of stochastic iterative dynamic programming algorithms. *Neural Computation*, 6(6), 1185-1201.
